In [ ]:
## Packages Import
import numpy             as np
import matplotlib.pyplot as plt

In [ ]:
## Utilities to the Lifting-Line Method

def simpson_weights(n, h):
    """
    Return the combined weights for all nodes according to the Simpson's
    quadrature rule
    """
    assert n % 2 == 0, "Simpson's rule requires an even number of subintervals"
    weight = np.zeros(n)
    weight[0] = weight[-1] = h/3
    weight[1:-1:2] = 4*h/3
    weight[2:-1:2] = 2*h/3
    return weight

def central_finite_diff(n, h, ghost=False):
    """
    Second-order accurate central finite difference method over the internal nodes
    of the domain, one-sided difference over boundary elements; extension of central
    difference over the entire domain in case of ghost cell technique
    """
    # Interior nodes + boundary nodes
    D = -1/(2*h) * np.diag(np.ones(n-1), -1) + \
        np.zeros((n, n)) + \
        1/(2*h) * np.diag(np.ones(n-1), 1)
    
    if (not ghost):
        # Overwrite boundary nodes
        # One-sided finite difference - 2nd order accurate
        D[0, :3]   = [-3,  4, -1]/(2*h)     # root node (forward finite diff)
        D[-1, -3:] = [ 1, -4,  3]/(2*h)     # tip node (backward finite diff)
        
    return D

def inter_diff_operator(h, zeta):
    """
    Build the integral-differential operator relating downwash velocity at point P
    to the circulation over the lifting line; it does not account for scalar
    multiplicative constants
                    M -> v(zP) = 1/(4π) * M @ gamma
    """
    N  = len(zeta)   # number of cells

    # Integral kernel matrix
    weight = simpson_weights(N, h)      # quadrature weights (N, 1)
    H = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if i != j:
                H[i, j] = weight[j] / (zeta[j] - zeta[i]) # TBD: check sign of denominator

    # Differential matrix
    D = central_finite_diff(N, h, ghost=True)   # (N, N)
    
    # Full intregral-differential matrix
    M = np.dot(H, D)                            # (N, N)
    return M

def dirichlet_bc(A, b, idx, value=0.0):
    """
    Impose boundary conditions of Dirichlet type at given index, corresponding to
    boundary cells, default condition is zero load
    """ 
    A[idx, :]   = 0.0
    A[idx, idx] = 1.0
    b[idx] = value

In [ ]:
## Other Utilities

def chord(z, c0, s, form="rectangular"):
    """
    Return chord length distribution along spanwise direction according to the
    desired planform shape 
    """
    match form:
        case "rectangular":
            c = c0 * np.ones(z.shape)
        case "elliptical":
            c = c0 * np.sqrt(1 - (z/s)**2)
    return c

def ghost_node_padding(a, val):
    """
    Add symmetric padding for extra-domain nodes (ghost nodes)
    """
    if type(val) == int:
        # Rcast assigned value to list type for generality
        val = [0]

    aG = np.append(val[0], a)
    aG = np.append(aG, val[-1])
    return aG

In [ ]:
## Parameters of a wing of ractangular planform
b = 10          # [m] wing span
s = b / 2       # [m] wing semi-span
c0 = 1          # [m] chord @ root
a0 = 0.1095     # [1/deg] lift curve slope
alpha = 6       # [deg] geometric angle of attack
alpha_L0 = -2   # [deg] angle of attack @ zero lift
U = 5           # [m/s] incident flow velocity

# Wing span discretization
N = 100     # number of cells - must be even!
z = np.linspace(-s, s, N+1)         # cell nodes     (N+1, 1)
dz = abs(z[1] - z[0])               # cells size

zeta = (z[1:] + z[:-1]) / 2                                     # cell centroids               (N,   1)
zG = ghost_node_padding(zeta, [zeta[0] - dz, zeta[-1] + dz])    # cell centroids + ghost nodes (N+2, 1)
M = inter_diff_operator(dz, zG)     # intergral-differential operator (N+2, N+2)

In [ ]:
## Test Intermediate
# Integral-differential operator against reference solution
g0 = 5                                  # circulation @ origin
gamma = g0*np.sqrt(1 - (zeta/s)**2)     # elliptical circulation
v_ref = -g0 / s                         # analytical solution

# Induced downwash velocity
v = 1/(4*np.pi) * np.dot(M[1:-1, 1:-1], gamma)    # numerical solution (no ghost nodes)
vs = 1/(4*np.pi) * np.dot(M[-1, 1:-1], gamma)     # downwash at wing tip

fig = plt.figure(figsize=(10, 4))
# Plot circulation distribution along span
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(zeta, gamma, c='r', linewidth=2, label="elliptical distr")
ax1.set_xlabel("Z")
ax1.set_ylabel(r'$\Gamma$')
ax1.legend()
ax1.set_title("Circulation Distribution")
ax1.grid(True)
# Plot downwash distribution along span
ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(zeta, v_ref * np.ones(N), c='k', linewidth=2, label="analytical sol")
ax2.plot(zeta, v - vs,                    linewidth=2, label="numerical sol")
ax2.set_xlabel("Z")
ax2.set_ylabel("v")
ax2.legend()
ax2.set_title("Velocity Downwash Distribution")
ax2.grid(True)
    
fig.tight_layout()
plt.show()

In [ ]:
## Test Lifting-Line Method
# Prescribe chord variation along span
c_shape = "elliptical"
c = chord(zeta, c0, s, c_shape)

# Linear system assembling
A = 1/(4*np.pi) * M - \
    ghost_node_padding(2/(np.rad2deg(a0)*c), 0) * np.diag(np.ones(len(zG)))    # linear system matrix
b = U*np.deg2rad(alpha_L0 - alpha) * np.ones(len(zG))    # right hand side
dirichlet_bc(A, b, [0, -1])      # apply free tip boundary conditions
# Solve linear system
x = np.linalg.solve(A, b)
 
fig = plt.figure(figsize=(6, 4))
# Plot circulation distribution along span
ax = fig.add_subplot()
ax.plot(zG, x, c='orange', linewidth=2, label=f"{c_shape} chord")
ax.set_xlabel("Z")
ax.set_ylabel(r'$\Gamma$')
ax.legend()
ax.set_title("Circulation Distribution")
ax.grid(True)